In [0]:
import requests
from datetime import datetime, timezone

HAZARD_API_URL = (
    "https://public.gis.lacounty.gov/public/rest/services/"
    "LACounty_Dynamic/Hazards/MapServer/2/query"
)

BRONZE_BASE_PATH = (
    "abfss://wildfire-data@stscewildfirevr01.dfs.core.windows.net/"
    "bronze/fire_hazard"
)

print("Configuration loaded.")

In [0]:
params = {
    "where": "1=1",
    "outFields": "*",
    "returnGeometry": "true",
    "outSR": "4326",
    "f": "geojson"
}

response = requests.get(
    HAZARD_API_URL,
    params=params,
    timeout=60
)

response.raise_for_status()

hazard_data = response.json()

print("API request successful")
print("Features returned:", len(hazard_data["features"]))

In [0]:
import json

run_timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

output_path = f"{BRONZE_BASE_PATH}/load_timestamp={run_timestamp}"

hazard_json = json.dumps(hazard_data)

raw_df = spark.createDataFrame(
    [(hazard_json, run_timestamp, len(hazard_data["features"]))],
    ["raw_json", "ingestion_timestamp", "feature_count"]
)

raw_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(output_path)

print("Bronze hazard ingestion successful")
print("Features stored:", len(hazard_data["features"]))
print("Path:", output_path)

In [0]:
check_df = spark.read.format("delta").load(output_path)

display(
    check_df.select(
        "ingestion_timestamp",
        "feature_count"
    )
)